# 01 — Filter betas by Spearman correlation with age

**Input:**
- `../data/raw/13datasets/betas.tsv` — ComBat-corrected methylation matrix
- `../data/raw/13datasets/Samplesheet.csv` — sample metadata with `Age`

**Output:**
- `outputs/01_filtered_betas/betas_rho{THRESHOLD}.tsv` — filtered betas (same format)
- `outputs/01_filtered_betas/CpG_spearman_correlations.tsv` — all CpG correlations
- `outputs/01_filtered_betas/CpG_pearson_correlations.tsv` — Pearson for reference
- `outputs/01_filtered_betas/filter_summary.txt` — what was done

**What it does:** for every CpG, compute `|Spearman(methylation, age)|`. Keep only CpGs where `|ρ| ≥ THRESHOLD`. Save correlation tables and the filtered beta matrix.

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────────────
BETA_TSV     ="input/Beta_ALL.tsv" #"../data/raw/13datasets/betas.tsv"
SAMPLESHEET  = "input/Samplesheet.csv"#"../data/raw/13datasets/Samplesheet.csv"
THRESHOLD    = 0.35     # |Spearman ρ| cutoff

OUT_DIR = Path("outputs/01_filtered_betas")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load betas and samplesheet

In [ ]:
t0 = time.time()
print(f"Loading {BETA_TSV} ...")
df = pd.read_csv(BETA_TSV, sep="\t", low_memory=False)
print(f"  Raw shape: {df.shape}")

# Detect orientation: CpGs as columns or rows?
cpg_cols = [c for c in df.columns if str(c).startswith("cg")]
if not cpg_cols:
    fc = df.columns[0]
    if df[fc].astype(str).head(20).str.startswith("cg").sum() > 10:
        print("  CpGs detected in rows (first column) -> transposing")
        df = df.set_index(fc).T.reset_index()
        df.rename(columns={df.columns[0]: "Sample_ID"}, inplace=True)
    else:
        iv = pd.Series(df.index.astype(str)).head(20)
        if iv.str.startswith("cg").sum() > 10:
            print("  CpGs in index -> transposing")
            df = df.T.reset_index()
            df.rename(columns={df.columns[0]: "Sample_ID"}, inplace=True)
    cpg_cols = [c for c in df.columns if str(c).startswith("cg")]

# Find sample ID column
idc = None
for c in ["Sample_ID", "ID_REF", "ID", "Sample", "geo_accession"]:
    if c in df.columns: idc = c; break

df_meth = df[cpg_cols].apply(pd.to_numeric, errors="coerce")
if idc:
    df_meth.index = df[idc].astype(str)
print(f"  Methylation matrix: {df_meth.shape[0]} samples × {df_meth.shape[1]:,} CpGs")

# Samplesheet
meta = pd.read_csv(SAMPLESHEET)
for c in meta.columns:
    if c.lower() == "age":
        meta = meta.rename(columns={c: "Age"})
meta["Age"] = pd.to_numeric(meta["Age"], errors="coerce")

mid = None
for c in ["Sample_ID", "ID", "Sample", "geo_accession", "ID_REF"]:
    if c in meta.columns: mid = c; break

if mid and idc:
    meta[mid] = meta[mid].astype(str)
    common = sorted(set(df_meth.index) & set(meta[mid]))
    print(f"  Matched {len(common)} samples")
    meta = meta.set_index(mid).loc[common].reset_index()
    df_meth = df_meth.loc[common].reset_index(drop=True)
    meta = meta.reset_index(drop=True)

# Drop samples without age
mask = meta["Age"].notna()
df_meth = df_meth.loc[mask].reset_index(drop=True)
meta = meta.loc[mask].reset_index(drop=True)
ages = meta["Age"].values
print(f"  Samples with valid age: {len(df_meth)}")
print(f"  Age range: {ages.min():.0f} – {ages.max():.0f}")
print(f"  [Loaded in {time.time()-t0:.1f}s]")

## Drop CpGs with too many missing values

In [ ]:
na_frac = df_meth.isna().mean()
ok = na_frac < 0.2
df_clean = df_meth.loc[:, ok]
dropped = (~ok).sum()
print(f"Dropped {dropped:,} CpGs with >20% NA")
print(f"{df_clean.shape[1]:,} CpGs remain")

## Compute Spearman and Pearson correlations

In [ ]:
print(f"Computing Spearman for {df_clean.shape[1]:,} CpGs ...")
t0 = time.time()
ages_s = pd.Series(ages, index=df_clean.index)
rho_sp = df_clean.corrwith(ages_s, method="spearman").dropna()
print(f"  Done [{time.time()-t0:.1f}s]")

print(f"Computing Pearson (for reference) ...")
t0 = time.time()
rho_pe = df_clean.corrwith(ages_s, method="pearson").dropna()
print(f"  Done [{time.time()-t0:.1f}s]")

# Save full correlation tables
pd.DataFrame({"CpG": rho_sp.index, "Spearman_rho": rho_sp.values}).to_csv(
    OUT_DIR / "CpG_spearman_correlations.tsv", sep="\t", index=False)
pd.DataFrame({"CpG": rho_pe.index, "Pearson_r": rho_pe.values}).to_csv(
    OUT_DIR / "CpG_pearson_correlations.tsv", sep="\t", index=False)
print(f"  Saved: CpG_spearman_correlations.tsv + CpG_pearson_correlations.tsv")

## Filter CpGs by threshold and save

In [ ]:
abs_rho = rho_sp.abs()
passing = abs_rho[abs_rho >= THRESHOLD]
n_pass = len(passing); n_total = len(abs_rho)
print(f"  {n_pass:,} / {n_total:,} CpGs pass |Spearman| >= {THRESHOLD} ({100*n_pass/n_total:.1f}%)")

if n_pass == 0:
    raise RuntimeError(f"No CpGs pass threshold {THRESHOLD}. Lower it and retry.")

print(f"\nTop 10 CpGs by |Spearman|:")
for cpg, val in passing.sort_values(ascending=False).head(10).items():
    print(f"  {cpg}: |rho| = {val:.4f}")

keep_cpgs = set(passing.index)

In [ ]:
# Reload original file and filter, preserving its exact format
print("Rebuilding filtered output in original orientation ...")
df_orig = pd.read_csv(BETA_TSV, sep="\t", low_memory=False)

orig_cpg_cols = [c for c in df_orig.columns if str(c).startswith("cg")]
if orig_cpg_cols:
    non_cpg = [c for c in df_orig.columns if not str(c).startswith("cg")]
    keep_cols = non_cpg + [c for c in orig_cpg_cols if c in keep_cpgs]
    df_out = df_orig[keep_cols]
else:
    fc = df_orig.columns[0]
    if df_orig[fc].astype(str).head(20).str.startswith("cg").sum() > 10:
        df_out = df_orig[df_orig[fc].astype(str).isin(keep_cpgs)]
    else:
        df_out = df_orig[df_orig.index.astype(str).isin(keep_cpgs)]

out_name = f"betas_rho{THRESHOLD:.2f}.tsv"
out_path = OUT_DIR / out_name
df_out.to_csv(out_path, sep="\t", index=False)
print(f"\n  Saved: {out_path}")
print(f"  Shape: {df_out.shape}")

# Summary file
with open(OUT_DIR / "filter_summary.txt", "w") as f:
    f.write(f"Input: {BETA_TSV}\n")
    f.write(f"Samplesheet: {SAMPLESHEET}\n")
    f.write(f"Threshold: |Spearman rho| >= {THRESHOLD}\n")
    f.write(f"Samples (with age): {len(df_meth)}\n")
    f.write(f"CpGs before NA filter: {df_meth.shape[1]}\n")
    f.write(f"CpGs after NA filter: {df_clean.shape[1]}\n")
    f.write(f"CpGs passing |rho| >= {THRESHOLD}: {n_pass}\n")
    f.write(f"Output file: {out_path.name}\n")
print(f"  Summary: filter_summary.txt")